# Exercice 6 — Solution

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.74.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.Ollama, 1.74.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.Qdrant, 1.74.0-preview"
#pragma warning disable SKEXP0070,SKEXP0001
using Microsoft.Extensions.AI; using Microsoft.Extensions.VectorData; using Microsoft.SemanticKernel; using Microsoft.SemanticKernel.Connectors.Qdrant; using Qdrant.Client;
using System.Security.Cryptography; using System.Text;
var endpoint=new Uri("http://localhost:11434");
var embed=Kernel.CreateBuilder().AddOllamaEmbeddingGenerator("nomic-embed-text",endpoint).Build().GetRequiredService<IEmbeddingGenerator<string,Embedding<float>>>();
var store=new QdrantVectorStore(new QdrantClient("localhost"),ownsClient:true);
ulong Id(string s)=>BitConverter.ToUInt64(MD5.HashData(Encoding.UTF8.GetBytes(s)),0);

In [ ]:
public sealed class Doc{ [VectorStoreKey] public ulong Id{get;set;} [VectorStoreData] public string Text{get;set;}="";
  [VectorStoreVector(768,DistanceFunction=DistanceFunction.CosineSimilarity)] public ReadOnlyMemory<float> Vector{get;set;} }

In [ ]:
var kernel=Kernel.CreateBuilder().AddOllamaChatCompletion("gemma3:4b",endpoint).Build();
var col=store.GetCollection<ulong,Doc>("ex6"); await col.EnsureCollectionExistsAsync();
var fait="Le VPN se configure via le portail interne vpn.cognitic.local avec vos identifiants AD.";
await col.UpsertAsync(new Doc{Id=Id(fait),Text=fait,Vector=(await embed.GenerateAsync(fait)).Vector});
var qv=(await embed.GenerateAsync("comment configurer le VPN ?")).Vector;
var passages=new List<string>(); await foreach(var r in col.SearchAsync(qv,top:2)) passages.Add(r.Record.Text);
var rep=await kernel.InvokePromptAsync("Reponds depuis le contexte.\nContexte: {{$c}}\nQuestion: {{$q}}",new(){["c"]=string.Join("\n",passages),["q"]="Comment configurer le VPN ?"});
Console.WriteLine(rep);